In [ ]:
import os
import pickle
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from src.data_preprocessing import load_data, get_features, get_numeric_categorical_features, build_preprocessor

data_path = '/kaggle/input/daotka/result.csv'
df = load_data(data_path)
X, y = get_features(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features, categorical_features = get_numeric_categorical_features(X_train)

preprocessor = build_preprocessor(numeric_features, categorical_features)

pca = TruncatedSVD(n_components=10)

# Лучшая модель
best_classifier = DecisionTreeClassifier(max_depth=5, criterion="entropy", random_state=42)

# Собираем финальный пайплайн для инференса
best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', pca),
    ('classifier', best_classifier)
])

# Обучаем лучшую модель
best_pipeline.fit(X_train, y_train)

model_filename = os.path.join("experiments", "DecisionTreeClassifier_PCA_EXP_11_best_model.pkl")
os.makedirs("experiments", exist_ok=True)
with open(model_filename, "wb") as f:
    pickle.dump(best_pipeline, f)
print(f"Лучшей модель сохранена в: {model_filename}")

y_pred = best_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nConfusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", class_report)